# Exporting the azure data into databricks 

In [0]:
%python
storage_account_name = "reatilproject"
container_name = "retail"
secret_key = "PASS_KEY" 

# Target the base customers folder
customers_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze/customers"

# Pass BOTH options directly into the reader
customers_df = spark.read \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", secret_key) \
    .option("recursiveFileLookup", "true") \
    .format("parquet") \
    .load(customers_path)

display(customers_df.limit(5))

customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28


### Reading the different data sources

In [0]:
%python
storage_account_name = "reatilproject"
container_name = "retail"
secret_key = "PASS_KEY" 

# Base path for your bronze layer
base_bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

# 1. Load Product Data
product_df = spark.read \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", secret_key) \
    .option("recursiveFileLookup", "true") \
    .format("parquet") \
    .load(f"{base_bronze_path}/product")

# 2. Load Store Data
store_df = spark.read \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", secret_key) \
    .option("recursiveFileLookup", "true") \
    .format("parquet") \
    .load(f"{base_bronze_path}/store")

# 3. Load Transaction Data
transaction_df = spark.read \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", secret_key) \
    .option("recursiveFileLookup", "true") \
    .format("parquet") \
    .load(f"{base_bronze_path}/transaction")

print("Products preview:")
display(product_df.limit(3))

print("Stores preview:")
display(store_df.limit(3))

print("Transactions preview:")
display(transaction_df.limit(3))

Products preview:


product_id,product_name,category,price
1,Wireless Mouse,Electronics,799.99
2,Bluetooth Speaker,Electronics,1299.49
3,Yoga Mat,Fitness,499.00


Stores preview:


store_id,store_name,location
1,City Mall Store,Mumbai
2,High Street Store,Delhi
3,Tech World Outlet,Bangalore


Transactions preview:


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01


# Silver layer

### Coverting data types

In [0]:
from pyspark.sql.functions import *
transaction_df=transaction_df.select(
    col('transaction_id').cast('int'),
    col('customer_id').cast('int'),
    col('product_id').cast('int'),
    col('store_id').cast('int'),
    col('transaction_date').cast('date'),
    col('quantity').cast('int')
)


In [0]:
product_df=product_df.select(
    col('product_id').cast('int'),
    col('price').cast('double'),
    col('product_name'),
    col('category')
)

In [0]:
store_df=store_df.select(
    col('store_id').cast('int'),
    col('store_name'),
    col('location')
)

In [0]:
customers_df=customers_df.select(
    col('customer_id').cast('int'),
    col('first_name'),
    col('last_name'),
    col('email'),
    col('phone'),
    col('city'),
    col('registration_date').cast('date')).dropDuplicates(['customer_id'])

In [0]:
silver_df=transaction_df \
    .join(customers_df,'customer_id')\
    .join(product_df,'product_id')\
    .join(store_df,'store_id')\
    .withColumn('total_amt',round(col('price')*col('quantity').cast('double'),2))

In [0]:
display(silver_df)

store_id,product_id,customer_id,transaction_id,transaction_date,quantity,first_name,last_name,email,phone,city,registration_date,price,product_name,category,store_name,location,total_amt
3,1,105,21,2024-10-02,5,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,799.99,Wireless Mouse,Electronics,Tech World Outlet,Bangalore,3999.95
5,6,121,15,2025-05-19,5,Sanjay,Patel,user121@example.com,9687654321,Chennai,2024-01-10,299.0,Water Bottle,Fitness,Mega Plaza,Chennai,1495.0
3,3,104,13,2025-05-04,4,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05,499.0,Yoga Mat,Fitness,Tech World Outlet,Bangalore,1996.0
1,5,108,12,2025-05-26,4,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19,149.0,Notebook Set,Stationery,City Mall Store,Mumbai,596.0
3,1,123,9,2024-10-08,2,Arjun,Yadav,user123@example.com,9665432109,Ahmedabad,2024-02-25,799.99,Wireless Mouse,Electronics,Tech World Outlet,Bangalore,1599.98
2,2,124,10,2024-08-27,5,Kavita,Sharma,user124@example.com,9654321098,Kolkata,2023-11-15,1299.49,Bluetooth Speaker,Electronics,High Street Store,Delhi,6497.45
5,8,109,17,2024-07-10,5,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01,399.0,Desk Organizer,Accessories,Mega Plaza,Chennai,1995.0
2,6,126,26,2024-09-21,2,Bhavna,Nair,user126@example.com,9632109876,Mumbai,2023-07-17,299.0,Water Bottle,Fitness,High Street Store,Delhi,598.0
4,1,103,18,2024-09-05,3,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10,799.99,Wireless Mouse,Electronics,Downtown Mini Store,Pune,2399.97
4,9,122,23,2025-04-30,2,Tina,Kapoor,user122@example.com,9676543210,Pune,2023-09-20,1999.0,Dumbbell Set,Fitness,Downtown Mini Store,Pune,3998.0


### Uploading into the Silver Layer

In [0]:
# 1. Target the path to your silver layer folder
silver_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

# 2. Write the DataFrame to Azure
silver_df.write \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", secret_key) \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Successfully written data to the Silver layer!")

Successfully written data to the Silver layer!


In [0]:
# 1. Read the clean silver data from Azure using your credentials
silver_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

silver_data_df = spark.read \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", secret_key) \
    .format("delta") \
    .load(silver_path)

# 2. Save it directly as a managed table in your catalog
silver_data_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("reatil_silver_cleaned")

print("Table 'reatil_silver_cleaned' successfully registered!")

Table 'reatil_silver_cleaned' successfully registered!


In [0]:
%sql
select * from reatil_silver_cleaned as re
limit 5;

store_id,product_id,customer_id,transaction_id,transaction_date,quantity,first_name,last_name,email,phone,city,registration_date,price,product_name,category,store_name,location,total_amt
3,1,105,21,2024-10-02,5,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,799.99,Wireless Mouse,Electronics,Tech World Outlet,Bangalore,3999.95
5,6,121,15,2025-05-19,5,Sanjay,Patel,user121@example.com,9687654321,Chennai,2024-01-10,299.0,Water Bottle,Fitness,Mega Plaza,Chennai,1495.0
3,3,104,13,2025-05-04,4,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05,499.0,Yoga Mat,Fitness,Tech World Outlet,Bangalore,1996.0
1,5,108,12,2025-05-26,4,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19,149.0,Notebook Set,Stationery,City Mall Store,Mumbai,596.0
3,1,123,9,2024-10-08,2,Arjun,Yadav,user123@example.com,9665432109,Ahmedabad,2024-02-25,799.99,Wireless Mouse,Electronics,Tech World Outlet,Bangalore,1599.98


## Grouping the columns required for the Gold Layer

In [0]:
from pyspark.sql.functions import sum, countDistinct, avg

gold_df = silver_df.groupBy(
"transaction_date",
"product_id", "product_name", "category",
"store_id", "store_name", "location"
).agg(
sum("quantity").alias("total_quantity_sold"),
sum("total_amt").alias("total_sales_amount"),
countDistinct("transaction_id").alias("number_of_transactions"),
avg("total_amt").alias("average_transaction_value"))

In [0]:
display(gold_df)

transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2024-10-02,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,5,3999.95,1,3999.95
2025-05-19,6,Water Bottle,Fitness,5,Mega Plaza,Chennai,5,1495.0,1,1495.0
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2025-05-26,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,4,596.0,1,596.0
2024-10-08,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
2024-07-10,8,Desk Organizer,Accessories,5,Mega Plaza,Chennai,5,1995.0,1,1995.0
2024-09-21,6,Water Bottle,Fitness,2,High Street Store,Delhi,2,598.0,1,598.0
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.97,1,2399.97
2025-04-30,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0


# Writing the Silver Layer back onto the Gold Layer

In [0]:
gold_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/gold"

# Dump the data files into the Azure 'gold' folder
gold_df.write \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", secret_key) \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_sales_summary")

print("Successfully dumped Gold data to Azure and registered!")

Successfully dumped Gold data to Azure and registered!
